# Exploratory Data Analysis — Ship GPS Data

Analiza danych GPS ze statków w ramach projektu **Marine Recognition** (Data Waves / Center of Marine Physics).

**Cel:** Zrozumienie struktury danych, rozkładów cech kinematycznych i charakterystyk poszczególnych stanów operacyjnych statku.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.dataset import load_gps_csv
from src.data.preprocessing import DataPreprocessor
from src.utils.config import OPERATION_ID_TO_STATE

sns.set_theme(style="whitegrid", font_scale=1.1)
pd.set_option("display.max_columns", 20)

STATE_COLORS = {"port_stay": "#2196F3", "anchor": "#FF9800", "adrift": "#F44336", "voyage": "#4CAF50"}

## 1. Wczytanie i preprocessing danych

In [ ]:
from pathlib import Path

df1 = load_gps_csv(Path("../data/raw/classified/Ship_Operation_example_dataset_classified.csv"))
df2 = load_gps_csv(Path("../data/raw/classified/Ship_Operation_example_dataset_classified_2.csv"))

raw = pd.concat([df1, df2], ignore_index=True)
print(f"Raw data: {len(raw)} rows")
print(f"Dataset 1: {len(df1)} rows ({df1['signaldate'].min()} — {df1['signaldate'].max()})")
print(f"Dataset 2: {len(df2)} rows ({df2['signaldate'].min()} — {df2['signaldate'].max()})")

preprocessor = DataPreprocessor()
df = preprocessor.process(raw)
df["state"] = df["operation_id"].map(OPERATION_ID_TO_STATE)

print(f"\nAfter preprocessing: {len(df)} rows")
df.head()

## 2. Rozkład stanów operacyjnych

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df["state"].value_counts()
colors = [STATE_COLORS[s] for s in counts.index]
axes[0].bar(counts.index, counts.values, color=colors, alpha=0.85)
for i, (state, count) in enumerate(counts.items()):
    axes[0].text(i, count + 200, f"{count}\n({count/len(df)*100:.1f}%)", ha="center", fontsize=9)
axes[0].set_title("Liczba punktów per stan")
axes[0].set_ylabel("Liczba punktów")

# Episodes
episodes = (df["state"] != df["state"].shift()).cumsum()
ep_info = df.groupby(episodes)["state"].first().value_counts()
ep_colors = [STATE_COLORS[s] for s in ep_info.index]
axes[1].bar(ep_info.index, ep_info.values, color=ep_colors, alpha=0.85)
for i, (state, count) in enumerate(ep_info.items()):
    axes[1].text(i, count + 0.3, str(count), ha="center", fontsize=10)
axes[1].set_title(f"Liczba epizodów per stan (total: {episodes.nunique()})")
axes[1].set_ylabel("Liczba epizodów")

plt.tight_layout()
plt.show()

## 3. Statystyki cech kinematycznych per stan

In [ ]:
kinematic_cols = ["sog_knots", "dcog_deg", "rot_deg_s", "acceleration_ms2", "distance_m"]
stats = df.groupby("state")[kinematic_cols].describe().round(3)
stats.T

## 4. Rozkłady SOG per stan

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, state in zip(axes.flat, ["port_stay", "anchor", "adrift", "voyage"]):
    data = df[df["state"] == state]["sog_knots"].dropna()
    ax.hist(data, bins=50, color=STATE_COLORS[state], alpha=0.75, edgecolor="white")
    ax.axvline(data.median(), color="black", linestyle="--", label=f"median={data.median():.2f}")
    ax.set_title(f"{state} (n={len(data)})")
    ax.set_xlabel("SOG (knots)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

plt.suptitle("Rozkład prędkości (SOG) per stan operacyjny", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Seria czasowa SOG/COG z kolorowaniem stanów

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

for state in ["port_stay", "anchor", "adrift", "voyage"]:
    mask = df["state"] == state
    axes[0].scatter(df.loc[mask, "signaldate"], df.loc[mask, "sog_knots"],
                    c=STATE_COLORS[state], s=1, label=state, alpha=0.7)
    axes[1].scatter(df.loc[mask, "signaldate"], df.loc[mask, "cog_deg"],
                    c=STATE_COLORS[state], s=1, alpha=0.7)

axes[0].set_ylabel("SOG (knots)")
axes[0].set_title("Prędkość i kurs w czasie (ground truth)")
axes[0].legend(markerscale=10, loc="upper right")
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel("COG (degrees)")
axes[1].set_xlabel("Time")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Korelacja cech kinematycznych

In [ ]:
corr_cols = ["sog_knots", "dcog_deg", "rot_deg_s", "acceleration_ms2", "distance_m"]
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[corr_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title("Macierz korelacji cech kinematycznych")
plt.tight_layout()
plt.show()

## 7. Boxploty SOG per stan

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, label in zip(axes, ["sog_knots", "dcog_deg", "acceleration_ms2"],
                            ["SOG (knots)", "dCOG (degrees)", "Acceleration (m/s²)"]):
    palette = [STATE_COLORS[s] for s in ["port_stay", "anchor", "adrift", "voyage"]]
    sns.boxplot(data=df, x="state", y=col, ax=ax, palette=palette,
                order=["port_stay", "anchor", "adrift", "voyage"],
                showfliers=False)
    ax.set_xlabel("")
    ax.set_ylabel(label)
    ax.set_title(label)

plt.suptitle("Rozkłady cech kinematycznych per stan (bez outlierów)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 8. Analiza długości epizodów

In [ ]:
episodes = (df["state"] != df["state"].shift()).cumsum()
ep_df = df.groupby(episodes).agg(
    state=("state", "first"),
    start=("signaldate", "first"),
    end=("signaldate", "last"),
    n_points=("state", "count"),
).reset_index(drop=True)
ep_df["duration_min"] = (ep_df["end"] - ep_df["start"]).dt.total_seconds() / 60

print(f"Total episodes: {len(ep_df)}")
print(f"\nDuration stats (minutes):")
print(ep_df.groupby("state")["duration_min"].describe().round(1))

fig, ax = plt.subplots(figsize=(12, 5))
for state in ["port_stay", "anchor", "adrift", "voyage"]:
    data = ep_df[ep_df["state"] == state]["duration_min"]
    ax.barh(state, data.mean(), color=STATE_COLORS[state], alpha=0.85)
    ax.text(data.mean() + 10, state, f"avg={data.mean():.0f} min (n={len(data)})", va="center")

ax.set_xlabel("Avg duration (minutes)")
ax.set_title("Średnia długość epizodów per stan")
plt.tight_layout()
plt.show()

## 9. Scatter: SOG vs dCOG per stan

Kluczowa wizualizacja — pokazuje jak stany rozdzielają się w przestrzeni prędkości i zmiany kursu.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for state in ["port_stay", "anchor", "adrift", "voyage"]:
    mask = df["state"] == state
    sample = df[mask].sample(min(2000, mask.sum()), random_state=42)
    ax.scatter(sample["sog_knots"], sample["dcog_deg"].abs(),
               c=STATE_COLORS[state], s=5, label=state, alpha=0.5)

ax.axvline(3.0, color="gray", linestyle="--", alpha=0.5, label="voyage threshold (3 kn)")
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.5, label="stationary threshold (0.5 kn)")
ax.set_xlabel("SOG (knots)")
ax.set_ylabel("|dCOG| (degrees)")
ax.set_title("Przestrzeń cech: SOG vs |dCOG|")
ax.legend(markerscale=4)
ax.set_xlim(-0.5, 20)
ax.set_ylim(-5, 180)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Podsumowanie

**Kluczowe obserwacje:**
- **30K punktów, 21 epizodów** — mały dataset, co ogranicza ML
- **Voyage** i **port_stay** dominują (>80% punktów), **adrift** jest rzadki
- SOG wyraźnie rozdziela stany: port <0.5 kn, anchor ~0, adrift 0.5-3 kn, voyage >3 kn
- dCOG jest szumowy ale pomaga rozróżnić adrift (chaotyczny kurs) od voyage (stabilny kurs)
- **21 epizodów** to za mało na generalizację ML — rule-based z wiedzą domenową jest bardziej stabilny